# LifeMate: baseline agent bằng text trên Kaggle

Transcript chuẩn + context + history + tool schemas → Qwen3-4B (`enable_thinking=False`) → runtime BTC.

**Mặc định chỉ chạy agent B0, chưa chạy audio.** Bật `RUN_AUDIO` sau khi đã đánh giá agent. Notebook đã kiểm tra cấu trúc trên local; chưa thực thi GPU, chưa có điểm model.

Gắn data cuộc thi và source dự án làm Kaggle Dataset; bật GPU và Internet cho lần tải model đầu. B0 dùng một T4; đo VRAM thực tế với history dài. Public test chưa có transcript chuẩn nên để giai đoạn audio sau.


In [ ]:
# Giữ Torch/NumPy của Kaggle; chỉ cài thư viện model nếu cần.
%pip install -q "transformers>=4.51.0,<5" "accelerate>=1.0,<2" "safetensors>=0.5,<1"


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
import torch

INPUT = Path('/kaggle/input')
source_candidates = sorted({p.parents[2] for p in INPUT.rglob('src/lifemate/run.py')})
data_candidates = sorted({p.parent.parent for p in INPUT.rglob('train.jsonl')
                          if p.parent.name == 'train' and
                          (p.parent.parent / 'public_test/public_test.jsonl').is_file()})
print('Source candidates:', source_candidates)
print('Data candidates:', data_candidates)
# Nếu có nhiều dataset phù hợp, thay hai dòng sau bằng đường dẫn bạn chọn.
assert len(source_candidates) == 1, 'Chọn SOURCE_ROOT từ danh sách ở trên'
assert len(data_candidates) == 1, 'Chọn DATA_ROOT từ danh sách ở trên'
SOURCE_ROOT = source_candidates[0]
DATA_ROOT = data_candidates[0]
WORK = Path('/kaggle/working/lifemate')
WORK.mkdir(parents=True, exist_ok=True)
ENV = dict(os.environ, PYTHONPATH=str(SOURCE_ROOT / 'src'), PYTHONHASHSEED='42')
assert torch.cuda.is_available(), 'Bật GPU trong Notebook settings'
print('GPU:', [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
ASR_DEVICE = 'cuda:0'
LLM_DEVICE = 'cuda:0'
RUN_AUDIO = False  # Giai đoạn hiện tại chỉ đo agent bằng transcript chuẩn
ASR_MODEL = 'vinai/PhoWhisper-medium'
LLM_MODEL = 'Qwen/Qwen3-4B'
# Có thể thay bằng thư mục model local đã chứa đủ tokenizer/config/weights.
# Với offline: ENV.update(HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')

def run_module(module, *args):
    command = [sys.executable, '-m', 'lifemate.' + module, *map(str, args)]
    subprocess.run(command, env=ENV, check=True)


## Audit và split cố định

Audit không sửa dữ liệu BTC. Chỉ replay nhãn train để kiểm tra, không gắn nhãn test. Split giữ hội thoại nguyên vẹn và gộp các yêu cầu mới trùng text; chưa bảo đảm tách speaker/template. Pilot 48 mẫu cân bằng ba nhóm để debug; cần đánh giá đầy đủ 309 dev sau đó.


In [ ]:
AUDIT = WORK / 'data_audit'
run_module('audit', '--data-root', DATA_ROOT, '--output', AUDIT)
PILOT_IDS = AUDIT / 'pilot_ids.json'
DEV_IDS = AUDIT / 'dev_ids.json'


## B0: agent với transcript chuẩn

B0 chỉ chạy trên train/dev. Nếu B0 yếu, sửa agent trước. Model tải lần đầu có thể mất thời gian/dung lượng; thời gian load không thuộc số đo từng mẫu. Các run dùng thư mục mới, không ghi đè kết quả cũ.


In [ ]:
B0 = WORK / 'b0_oracle_pilot'
run_module('run', '--data-root', DATA_ROOT, '--split', 'train', '--ids', PILOT_IDS,
           '--mode', 'oracle', '--llm-model', LLM_MODEL,
           '--llm-device', LLM_DEVICE, '--output', B0)
run_module('evaluate', '--data-root', DATA_ROOT, '--run', B0)


## B1: audio — để thử nghiệm sau

Cell dưới chỉ chạy khi `RUN_AUDIO=True`. Giữ nguyên agent và tập pilot khi so với B0 để xem lỗi nhận dạng ảnh hưởng đến hành động thế nào.


In [ ]:
B1 = None
if RUN_AUDIO:
    B1 = WORK / 'b1_audio_pilot'
    run_module('run', '--data-root', DATA_ROOT, '--split', 'train', '--ids', PILOT_IDS,
               '--asr-model', ASR_MODEL, '--llm-model', LLM_MODEL,
               '--asr-device', ASR_DEVICE, '--llm-device', LLM_DEVICE, '--output', B1)
    run_module('evaluate', '--data-root', DATA_ROOT, '--run', B1)
else:
    print('Đang tập trung baseline agent B0; chưa tải/chạy PhoWhisper.')


In [ ]:
for name, directory in [('B0 oracle', B0), ('B1 audio', B1)]:
    if directory is None:
        continue
    result = json.loads((directory / 'evaluation.json').read_text())
    print(name, json.dumps({k: result[k] for k in [
        'samples', 'state_match_rate', 'mean_exact_call_f1',
        'state_gated_exact_f1_percent', 'asr', 'latency_seconds', 'stop_reasons'
    ]}, ensure_ascii=False, indent=2))
# Exact-call diagnostic KHÔNG phải điểm fuzzy chính thức.
# WER của B0 oracle bằng 0 không phải bằng chứng về ASR.


## Chạy đủ 309 dev sau pilot

Bật `RUN_FULL_DEV` sau khi xem trace trên pilot. Khi `RUN_AUDIO=False`, chỉ B0 chạy trên full dev. Điểm full dev dùng phân phối tự nhiên; pilot 48 mẫu cân bằng chỉ để debug.

Public test để sau khi có audio pipeline. Hai T4 không tương đương phần cứng A30; đo một GPU và offline trước bản nộp cuối. Notebook chưa có Docker đã xác minh.


In [ ]:
RUN_FULL_DEV = False
RUN_PUBLIC = False

if RUN_FULL_DEV:
    dev_runs = [('oracle', WORK / 'b0_oracle_dev')]
    if RUN_AUDIO:
        dev_runs.append(('audio', WORK / 'b1_audio_dev'))
    for mode, output in dev_runs:
        run_module('run', '--data-root', DATA_ROOT, '--split', 'train', '--ids', DEV_IDS,
                   '--mode', mode, '--asr-model', ASR_MODEL, '--llm-model', LLM_MODEL,
                   '--asr-device', ASR_DEVICE, '--llm-device', LLM_DEVICE, '--output', output)
        run_module('evaluate', '--data-root', DATA_ROOT, '--run', output)

if RUN_PUBLIC:
    assert RUN_AUDIO, 'Public test cần audio pipeline; hiện chỉ chạy agent B0 trên train/dev.'
    PUBLIC_RUN = WORK / 'public_v1'
    run_module('run', '--data-root', DATA_ROOT, '--split', 'public_test',
               '--asr-model', ASR_MODEL, '--llm-model', LLM_MODEL,
               '--asr-device', ASR_DEVICE, '--llm-device', LLM_DEVICE, '--output', PUBLIC_RUN)
    validate_env = dict(ENV, PYTHONPATH=str(DATA_ROOT / 'public_test'))
    subprocess.run([sys.executable, '-m', 'zaic_tools', '--split', 'public_test',
                    '--root', str(DATA_ROOT), '--validate', str(PUBLIC_RUN / 'submission.jsonl')],
                   env=validate_env, check=True)
    rows = [json.loads(l) for l in (PUBLIC_RUN / 'submission.jsonl').read_text().splitlines()]
    expected = [json.loads(l)['id'] for l in (DATA_ROOT / 'public_test/public_test.jsonl').read_text().splitlines()]
    assert len(rows) == len(expected) and {r['id'] for r in rows} == set(expected)
    print('File cần nộp:', PUBLIC_RUN / 'submission.jsonl')
    print('Kiểm tra diagnostics để xử lý timeout/error trước khi nộp; schema hợp lệ không chứng minh đáp án đúng.')
